# 🇬🇧 UK Retail Performance Hub
## Phase 1 | Notebook 3 of 4 — Load to Database

**What this notebook does:**
- Loads the cleaned parquet files into a SQLite database
- Creates 5 analytical views that Power BI reads from directly
- Exports views as CSVs for easy Power BI connection
- Verifies everything loaded correctly

**Prerequisites:** Run `02_clean.ipynb` first

---

### Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
ROOT          = Path('/content/drive/MyDrive/uk-retail-performance-hub')
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = PROCESSED_DIR / 'retail_hub.db'

print(f'✓ Drive mounted')
print(f'  Database will be at: {DB_PATH}')

### Cell 2 — Import libraries

In [ ]:
import sqlite3
import pandas as pd

print('✓ Libraries imported')

### Cell 3 — Load cleaned data files

In [ ]:
uci_path = PROCESSED_DIR / 'uci_clean.parquet'
ons_path = PROCESSED_DIR / 'ons_clean.parquet'

if not uci_path.exists():
    raise FileNotFoundError(f'UCI clean file not found at {uci_path}. Run 02_clean.ipynb first.')

uci_df = pd.read_parquet(uci_path)
print(f'✓ UCI loaded: {len(uci_df):,} rows')

if ons_path.exists():
    ons_df = pd.read_parquet(ons_path)
    print(f'✓ ONS loaded: {len(ons_df):,} rows')
else:
    ons_df = None
    print('⚠ ONS file not found — creating empty table. Market benchmark view will be empty.')

### Cell 4 — Create database and load tables

In [ ]:
def load_table(conn, df, table_name):
    """Load a DataFrame into SQLite, replacing if it already exists."""
    # Convert any datetime columns to strings for SQLite
    df = df.copy()
    for col in df.select_dtypes(include='datetime').columns:
        df[col] = df[col].astype(str)
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    count = pd.read_sql(f'SELECT COUNT(*) as n FROM {table_name}', conn)['n'][0]
    print(f'  ✓ {table_name:<20} {count:>10,} rows loaded')

conn = sqlite3.connect(DB_PATH)
print('Loading tables into database...')

load_table(conn, uci_df, 'transactions')

if ons_df is not None:
    load_table(conn, ons_df, 'ons_rsi')
else:
    conn.execute('''
        CREATE TABLE IF NOT EXISTS ons_rsi (
            Period TEXT, RSI_Value REAL, YearMonth TEXT,
            Year INTEGER, Month INTEGER, RSI_MoM_Pct REAL
        )
    ''')
    print('  ✓ ons_rsi              empty table created')

conn.commit()
print('\n✓ Tables loaded')

### Cell 5 — Create analytical views
These 5 views are what Power BI reads from. Each one is pre-aggregated so your dashboard loads fast.

In [ ]:
VIEWS = {

    'vw_monthly_revenue': '''
        CREATE VIEW IF NOT EXISTS vw_monthly_revenue AS
        SELECT
            YearMonth,
            Year,
            Month,
            COUNT(DISTINCT InvoiceNo)  AS OrderCount,
            COUNT(DISTINCT CustomerID) AS ActiveCustomers,
            ROUND(SUM(Revenue), 2)     AS TotalRevenue,
            SUM(Quantity)              AS UnitsSold,
            ROUND(SUM(CASE WHEN IsUK = 1 THEN Revenue ELSE 0 END), 2) AS UKRevenue,
            ROUND(SUM(CASE WHEN IsUK = 0 THEN Revenue ELSE 0 END), 2) AS IntlRevenue
        FROM transactions
        GROUP BY YearMonth, Year, Month
        ORDER BY YearMonth
    ''',

    'vw_customer_summary': '''
        CREATE VIEW IF NOT EXISTS vw_customer_summary AS
        SELECT
            CustomerID,
            Country,
            IsUK,
            CohortMonth,
            COUNT(DISTINCT InvoiceNo)     AS TotalOrders,
            ROUND(SUM(Revenue), 2)         AS TotalRevenue,
            ROUND(AVG(Revenue), 2)         AS AvgOrderValue,
            MIN(InvoiceDate)               AS FirstPurchase,
            MAX(InvoiceDate)               AS LastPurchase,
            CAST(JULIANDAY('2011-12-09') - JULIANDAY(MAX(InvoiceDate)) AS INTEGER) AS DaysSinceLastPurchase,
            COUNT(DISTINCT YearMonth)      AS ActiveMonths
        FROM transactions
        GROUP BY CustomerID, Country, IsUK, CohortMonth
    ''',

    'vw_product_performance': '''
        CREATE VIEW IF NOT EXISTS vw_product_performance AS
        SELECT
            ProductCategory,
            StockCode,
            Description,
            COUNT(DISTINCT InvoiceNo)  AS OrderCount,
            SUM(Quantity)              AS UnitsSold,
            ROUND(SUM(Revenue), 2)     AS TotalRevenue,
            ROUND(AVG(UnitPrice), 2)   AS AvgUnitPrice,
            COUNT(DISTINCT CustomerID) AS UniqueCustomers
        FROM transactions
        GROUP BY ProductCategory, StockCode, Description
        ORDER BY TotalRevenue DESC
    ''',

    'vw_market_vs_company': '''
        CREATE VIEW IF NOT EXISTS vw_market_vs_company AS
        SELECT
            m.YearMonth,
            m.TotalRevenue     AS CompanyRevenue,
            m.OrderCount,
            m.ActiveCustomers,
            o.RSI_Value        AS ONS_RSI,
            o.RSI_MoM_Pct      AS ONS_MoM_Change
        FROM vw_monthly_revenue m
        LEFT JOIN ons_rsi o ON m.YearMonth = o.YearMonth
        ORDER BY m.YearMonth
    ''',

    'vw_cohort_retention': '''
        CREATE VIEW IF NOT EXISTS vw_cohort_retention AS
        SELECT
            CohortMonth,
            YearMonth AS ActivityMonth,
            COUNT(DISTINCT CustomerID) AS ActiveCustomers,
            CAST(
                (SUBSTR(YearMonth,1,4) * 12 + CAST(SUBSTR(YearMonth,6,2) AS INT))
                - (SUBSTR(CohortMonth,1,4) * 12 + CAST(SUBSTR(CohortMonth,6,2) AS INT))
            AS INTEGER) AS MonthOffset
        FROM transactions
        GROUP BY CohortMonth, YearMonth
        ORDER BY CohortMonth, YearMonth
    ''',
}

cursor = conn.cursor()
print('Creating analytical views...')

for view_name, sql in VIEWS.items():
    cursor.execute(f'DROP VIEW IF EXISTS {view_name}')
    cursor.execute(sql)
    count = pd.read_sql(f'SELECT COUNT(*) as n FROM {view_name}', conn)['n'][0]
    print(f'  ✓ {view_name:<30} {count:>6,} rows')

conn.commit()
print('\n✓ All views created')

### Cell 6 — Export views to CSV for Power BI
Power BI connects to CSVs very easily. These 5 files become your Power BI data model.

In [ ]:
csv_dir = ROOT / 'data' / 'powerbi'
csv_dir.mkdir(parents=True, exist_ok=True)

exports = [
    ('vw_monthly_revenue',     'monthly_revenue.csv'),
    ('vw_customer_summary',    'customer_summary.csv'),
    ('vw_product_performance', 'product_performance.csv'),
    ('vw_market_vs_company',   'market_vs_company.csv'),
    ('vw_cohort_retention',    'cohort_retention.csv'),
]

print('Exporting views to CSV for Power BI...')
for view_name, filename in exports:
    df_export = pd.read_sql(f'SELECT * FROM {view_name}', conn)
    out_path  = csv_dir / filename
    df_export.to_csv(out_path, index=False)
    print(f'  ✓ {filename:<35} {len(df_export):,} rows → {out_path}')

conn.close()

print(f'\n✓ All CSVs saved to {csv_dir}')
print('\nIn Power BI Desktop:')
print('  Get Data → Text/CSV → navigate to your Google Drive folder')
print('  Load each of the 5 CSV files above')

### Cell 7 — Database summary
Final confirmation of everything that was built.

In [ ]:
# Reconnect to read final state
conn = sqlite3.connect(DB_PATH)

print('=' * 55)
print('  PHASE 1 COMPLETE — DATABASE SUMMARY')
print('=' * 55)

summary = pd.read_sql('''
    SELECT
        COUNT(DISTINCT CustomerID) AS UniqueCustomers,
        COUNT(DISTINCT StockCode)  AS UniqueProducts,
        COUNT(DISTINCT InvoiceNo)  AS TotalOrders,
        ROUND(SUM(Revenue), 2)     AS TotalRevenue,
        MIN(InvoiceDate)           AS EarliestDate,
        MAX(InvoiceDate)           AS LatestDate
    FROM transactions
''', conn)

for col in summary.columns:
    print(f'  {col:<22}: {summary[col][0]}')

conn.close()

print('\n' + '=' * 55)
print('  ✓ Run 04_refresh.ipynb to see automated scheduling')
print('  ✓ Connect Power BI to data/powerbi/ CSV files')
print('=' * 55)